<a href="https://colab.research.google.com/github/Zuhair0000/Research_2/blob/main/paper_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Libraries**

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import tensorflow as tf
import random

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

# **Load Dataset**

In [ ]:
columns = ['engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + \
          [f'sensor_{i}' for i in range(1, 22)]

In [ ]:
train_df = pd.read_csv('/content/train_FD001.txt', sep='\s+', header=None, names=columns)
train_df.head()

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_628/1892888385.py:1: SyntaxWarning: invalid escape sequence '\s'
  train_df = pd.read_csv('/content/train_FD001.txt', sep='\s+', header=None, names=columns)


,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


# **Preprocessing**

## Calculating the RUL

In [ ]:
max_cycles = train_df.groupby('engine_id')['cycle'].max().reset_index()
max_cycles.rename(columns={'cycle': 'max_cycle'}, inplace=True)

train_df = train_df.merge(max_cycles, on=['engine_id'], how='left')

train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']

train_df.drop('max_cycle', axis=1, inplace=True)

print(f"Data Shape: {train_df.shape}")
print(train_df[['engine_id', 'cycle', 'RUL']].head(10))

Data Shape: (20631, 27)
   engine_id  cycle  RUL
0          1      1  191
1          1      2  190
2          1      3  189
3          1      4  188
4          1      5  187
5          1      6  186
6          1      7  185
7          1      8  184
8          1      9  183
9          1     10  182


In [ ]:
train_df = train_df.drop(columns=['op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19'], axis=1)

In [ ]:
train_df.head()

,engine_id,cycle,sensor_2,sensor_3,sensor_4,sensor_6,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,RUL
0,1,1,641.82,1589.70,1400.60,21.61,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190,191
1,1,2,642.15,1591.82,1403.14,21.61,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236,190
2,1,3,642.35,1587.99,1404.20,21.61,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442,189
3,1,4,642.35,1582.79,1401.87,21.61,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739,188
4,1,5,642.37,1582.85,1406.22,21.61,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044,187


## Exponential Smoothing

In [ ]:
sensor_columns = [col for col in train_df.columns if 'sensor' in col]

smoothed_sensors = train_df.groupby('engine_id')[sensor_columns].ewm(alpha=0.2).mean().reset_index(level=0, drop=True)

train_df[sensor_columns] = smoothed_sensors

print(f"Cleaned and Smoothed Data Shpae: {train_df.shape}")
train_df.head()

Cleaned and Smoothed Data Shpae: (20631, 18)


,engine_id,cycle,sensor_2,sensor_3,sensor_4,sensor_6,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,RUL
0,1,1,641.820000,1589.700000,1400.600000,21.61,554.360000,2388.060000,9046.190000,47.470000,521.660000,2388.020000,8138.620000,8.419500,392.000000,39.060000,23.419000,191
1,1,2,642.003333,1590.877778,1402.011111,21.61,554.021111,2388.048889,9045.012222,47.481111,522.004444,2388.047778,8134.658889,8.426333,392.000000,39.026667,23.421556,190
2,1,3,642.145410,1589.694262,1402.908197,21.61,554.119016,2388.061639,9048.261311,47.394590,522.174754,2388.040492,8134.073279,8.422836,391.180328,38.995246,23.389852,189
3,1,4,642.214715,1587.355420,1402.556504,21.61,554.231138,2388.078022,9048.674146,47.304959,522.406883,2388.053875,8133.990867,8.404328,391.457995,38.956206,23.384449,188
4,1,5,642.260909,1586.015159,1403.646311,21.61,554.162380,2388.072661,9050.600566,47.297535,522.342366,2388.049748,8133.934089,8.411786,391.916706,38.939486,23.390384,187


In [ ]:
train_df[sensor_columns].head()

,sensor_2,sensor_3,sensor_4,sensor_6,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21
0,641.820000,1589.700000,1400.600000,21.61,554.360000,2388.060000,9046.190000,47.470000,521.660000,2388.020000,8138.620000,8.419500,392.000000,39.060000,23.419000
1,642.003333,1590.877778,1402.011111,21.61,554.021111,2388.048889,9045.012222,47.481111,522.004444,2388.047778,8134.658889,8.426333,392.000000,39.026667,23.421556
2,642.145410,1589.694262,1402.908197,21.61,554.119016,2388.061639,9048.261311,47.394590,522.174754,2388.040492,8134.073279,8.422836,391.180328,38.995246,23.389852
3,642.214715,1587.355420,1402.556504,21.61,554.231138,2388.078022,9048.674146,47.304959,522.406883,2388.053875,8133.990867,8.404328,391.457995,38.956206,23.384449
4,642.260909,1586.015159,1403.646311,21.61,554.162380,2388.072661,9050.600566,47.297535,522.342366,2388.049748,8133.934089,8.411786,391.916706,38.939486,23.390384


## Scaling the Data

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
train_df[sensor_columns] = scaler.fit_transform(train_df[sensor_columns])

train_df.head()

,engine_id,cycle,sensor_2,sensor_3,sensor_4,sensor_6,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,RUL
0,1,1,0.150847,0.491234,0.304222,1.0,0.768933,0.359741,0.091503,0.422258,0.644213,0.229723,0.200097,0.443665,0.284632,0.770201,0.771027,191
1,1,2,0.225591,0.525921,0.336695,1.0,0.691402,0.331190,0.085245,0.430947,0.744663,0.300625,0.176827,0.477653,0.284632,0.733725,0.776076,190
2,1,3,0.283515,0.491065,0.357339,1.0,0.713800,0.363953,0.102508,0.363291,0.794330,0.282028,0.173387,0.460258,0.167980,0.699341,0.713443,189
3,1,4,0.311771,0.422185,0.349246,1.0,0.739452,0.406049,0.104702,0.293203,0.862026,0.316189,0.172903,0.368202,0.207496,0.656621,0.702766,188
4,1,5,0.330604,0.382714,0.374325,1.0,0.723721,0.392273,0.114938,0.287397,0.843210,0.305653,0.172569,0.405298,0.272778,0.638324,0.714492,187


In [ ]:
def create_sequence(df, sequence_length, feature_columns):
  X, y = [], []

  for engine_id, group in df.groupby('engine_id'):
    data_matrix = group[feature_columns].values
    labels = group['RUL'].values

    for i in range(len(data_matrix) - sequence_length):
      X.append(data_matrix[i: i + sequence_length])
      y.append(labels[i + sequence_length - 1])


  return np.array(X), np.array(y)


sequence_length = 30
X_train, y_train = create_sequence(train_df, sequence_length, sensor_columns)

print(f"X_train (3D Tensor) shape: {X_train.shape}")
print(f"y_train (3D Tensor) shape: {y_train.shape}")

X_train (3D Tensor) shape: (17631, 30, 15)
y_train (3D Tensor) shape: (17631,)


# **Architecting the Neural Network**

In [ ]:
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, GlobalAveragePooling1D, Multiply, Activation, Softmax
from tensorflow.keras.models import Model

inputs = Input(shape=(sequence_length, X_train.shape[2]))


spatial_scores = Dense(X_train.shape[2], activation='softmax')(inputs)

spatial_out = Multiply()([inputs, spatial_scores])

lstm_out = Bidirectional(LSTM(64, return_sequences=True))(spatial_out)

attention_scores = Dense(1, activation='tanh')(lstm_out)

attention_weights = Softmax(axis=1)(attention_scores)

attention_out = Multiply()([lstm_out, attention_weights])

x = GlobalAveragePooling1D()(attention_out)

outputs = Dense(1)(x)

model = Model(inputs, outputs)
model.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_15      │ (None, 30, 15)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_36 (Dense)    │ (None, 30, 15)    │        240 │ input_layer_15[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_23         │ (None, 30, 15)    │          0 │ input_layer_15[0… │
│ (Multiply)          │                   │            │ dense_36[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_14    │ (None, 30, 128)   │     40,960 │ multiply_23[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_37 (Dense)    │ (None, 30, 1)     │        129 │ bidirectional_14… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax_10          │ (None, 30, 1)     │          0 │ dense_37[0][0]    │
│ (Softmax)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_24         │ (None, 30, 128)   │          0 │ bidirectional_14… │
│ (Multiply)          │                   │            │ softmax_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ multiply_24[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_38 (Dense)    │ (None, 1)         │        129 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 41,458 (161.95 KB)

 Trainable params: 41,458 (161.95 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
max_rul = 125
train_df['RUL'] = train_df['RUL'].clip(upper=max_rul)

sequence_length = 30
X_train, y_train = create_sequence(train_df, sequence_length, sensor_columns)

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='mse',
              metrics=['mse'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


print('Starting training...')

history = model.fit(X_train, y_train,
                    epochs=100,
                    batch_size=64,
                    validation_split=0.2,
                    callbacks=[early_stop])

Starting training...
Epoch 1/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - loss: 8111.4409 - mse: 8111.4409 - val_loss: 8634.8740 - val_mse: 8634.8740
Epoch 2/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 7879.9648 - mse: 7879.9648 - val_loss: 8430.2764 - val_mse: 8430.2764
Epoch 3/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 7688.2114 - mse: 7688.2114 - val_loss: 8236.3135 - val_mse: 8236.3135
Epoch 4/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 7505.9551 - mse: 7505.9551 - val_loss: 8049.5576 - val_mse: 8049.5576
Epoch 5/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 7330.2456 - mse: 7330.2456 - val_loss: 7867.9727 - val_mse: 7867.9727
Epoch 6/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 7159.2739 - mse: 7159.2739 - val_loss: 7690.7183 - val_mse: 7690.7183
Epoch 7/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 6992.4590 - mse: 6992.4590 - val_loss: 7517.4893 - val_mse: 7517.4893
Epoch 8/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step -

In [ ]:
training_loss = history.history['loss']
validation_loss = history.history['val_loss']
epochs_range = range(1, len(training_loss) + 1)

# 2. Create the plot
plt.figure(figsize=(10, 6))
plt.plot(epochs_range, training_loss, label='Training Loss (MSE)', color='blue')
plt.plot(epochs_range, validation_loss, label='Validation Loss (MSE)', color='red', linestyle='--')

# 3. Add labels and title
plt.title('Dual-Attention Bi-LSTM Training Curve')
plt.xlabel('Epochs')
plt.ylabel('Mean Squared Error (MSE)')
plt.legend()
plt.grid(True)
plt.show()

# **Testing**

In [ ]:
test_df = pd.read_csv('test_FD001.txt', sep=r'\s+', header=None, names=columns)

test_df.drop(columns=['op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19'],
             axis=1,
             inplace=True)

smoothed_test = test_df.groupby('engine_id')[sensor_columns].ewm(alpha=0.2).mean().reset_index(level=0, drop=True)

test_df[sensor_columns] = smoothed_test

test_df[sensor_columns] = scaler.transform(test_df[sensor_columns])


X_test = []

for engine_id, group in test_df.groupby('engine_id'):
  data_matrix = group[sensor_columns].values

  if len(data_matrix) >= sequence_length:
    X_test.append(data_matrix[-sequence_length:])
  else:
    pad_length = sequence_length - len(data_matrix)
    pad_matrix = np.tile(data_matrix[0], (pad_length, 1))
    padded_data = np.vstack([pad_matrix, data_matrix])
    X_test.append(padded_data)

X_test = np.stack(X_test).astype(np.float32)


y_test = pd.read_csv('RUL_FD001.txt', sep=r'\s+', header=None, names=['RUL']).values
y_test = np.clip(y_test, a_min=None, a_max=125).astype(np.float32)


test_loss, test_mse = model.evaluate(X_test, y_test, verbose=0)
final_rmse = np.sqrt(test_mse)

print(f"Testing Data Shape: {X_test.shape}")
print(f"Final Test RMSE: {final_rmse:.2f} cycles")